<a href="https://colab.research.google.com/github/Kartikiscalm/IML-LAB-/blob/labs/Lab6_Evaluation_CrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab Practical: Cross-Validation and Evaluation

**Course:** Introduction to Machine Learning

# How to use this notebook

*  Read every Markdown cell before running the Code cell below it. Run cells in order (top to bottom).

*  Cells marked **EXERCISE** or containing TODO markers are where you write code.

*  These exercises are woven throughout the notebook — you must complete each one before moving on, because later cells may depend on your work.

*  Answer every Question in the text cell provided. Your reasoning matters as much as your code.

*  In Colab: run a cell with Shift + Enter. If something breaks, use Runtime → Restart and run all


## Prerequisites

Before diving into this lab practical, ensure you have a solid understanding of the following:

*   **Basic Python Programming:** Familiarity with Python syntax, data structures (lists, dictionaries), and control flow (loops, conditionals).
*   **NumPy:** Knowledge of array manipulation and basic mathematical operations using NumPy.
*   **Pandas:** Understanding of DataFrames, data loading, basic data manipulation, and cleaning.
*   **Machine Learning Fundamentals:** Concepts like supervised learning, training data, testing data, overfitting, and underfitting.
*   **Basic Statistics:** Mean, variance, standard deviation, and data distributions.
*   **Scikit-learn Basics:** How to import models, train them, and make predictions (e.g., `fit()`, `predict()`).

## Learning Objectives

Upon completing this lab practical, you will be able to:

1.  **Define Cross-Validation:** Understand what cross-validation is and its fundamental purpose in machine learning.
2.  **Understand the Need for Cross-Validation:** Explain why a simple train-test split might not be sufficient for robust model evaluation.
3.  **Implement K-Fold Cross-Validation:** Apply K-Fold cross-validation using `sklearn.model_selection` to evaluate model performance.
4.  **Implement Stratified K-Fold Cross-Validation and Leave One Out Cross-Validation:** Utilize Stratified K-Fold and LOOCV for classification tasks to maintain class proportions across folds.
5.  **Compare Different Cross-Validation Strategies:** Understand the advantages and disadvantages of different cross-validation techniques.
6.  **Evaluate Models Across Folds:** Calculate and interpret various performance metrics (e.g., accuracy, precision, recall, F1-score, RMSE) for each fold and average them.
7.  **Apply Cross-Validation in Hyperparameter Tuning:** Integrate cross-validation with hyperparameter tuning techniques like `GridSearchCV` to select optimal model parameters.
8.  **Identify and Mitigate Overfitting/Underfitting:** Use cross-validation results to diagnose and address issues of model overfitting or underfitting.
9.  **Model evaluation Metrics:**  Includes accuracy, precision, recall, F1-score, and the AUC-ROC curve to measure correct and incorrect class labels.


# Part-1 : Evaluation

In Part 2, we will delve into evaluation in machine learning. There are two primary types of evaluation metrics:

1.  **Regression Metrics:** These metrics are used to measure numerical prediction errors. Key regression metrics include:
    *   **Mean Absolute Error (MAE):** The average of the absolute differences between predicted and actual values. It measures the average magnitude of the errors without considering their direction.
        *   Formula: $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
    *   **Mean Squared Error (MSE):** The average of the squared differences between predicted and actual values. It penalizes larger errors more heavily.
        *   Formula: $MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
    *   **Root Mean Squared Error (RMSE):** The square root of the MSE. It provides an error measurement in the same units as the target variable, making it more interpretable than MSE.
        *   Formula: $RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$

2.  **Classification Metrics:** These metrics are used to evaluate the performance of classification models, which predict categorical outcomes. Key classification metrics include:
    *   **Accuracy:** The proportion of correctly classified instances (both true positives and true negatives) out of the total number of instances.
        *   Formula: $Accuracy = \frac{TP + TN}{TP + TN + FP + FN}$
    *   **Precision:** The proportion of true positive predictions that were actually correct. It answers: "Of all instances predicted as positive, how many were actually positive?"
        *   Formula: $Precision = \frac{TP}{TP + FP}$
    *   **Recall (Sensitivity):** The proportion of actual positive instances that were correctly identified. It answers: "Of all actual positive instances, how many did the model correctly identify?"
        *   Formula: $Recall = \frac{TP}{TP + FN}$
    *   **F1-Score:** The harmonic mean of Precision and Recall. It provides a single score that balances both metrics, especially useful when there's an uneven class distribution.
        *   Formula: $F1-Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}$
    *   **AUC-ROC Curve (Area Under the Receiver Operating Characteristic Curve):** A plot showing the performance of a classification model at all classification thresholds. AUC measures the entire 2D area underneath the entire ROC curve from (0,0) to (1,1). A higher AUC indicates a better model performance in distinguishing between positive and negative classes.

### Understanding the Confusion Matrix (Binary Classification)

To better understand terms like True Positives (TP), False Positives (FP), False Negatives (FN), and True Negatives (TN), a **confusion matrix** is an invaluable tool. It's a table that summarizes the performance of a classification algorithm.

For a binary classification problem, where there are two classes (e.g., 'Positive' and 'Negative' or 'Yes' and 'No'), the confusion matrix looks like this:

```
                          Predicted Class
                       |  Positive   |   Negative  |
-----------------------------------------------------
Actual Class: Positive   |  True Positive (TP) | False Negative (FN) |
Actual Class: Negative   | False Positive (FP) | True Negative (TN)  |
```

Let's break down each term:

*   **True Positive (TP):** The model correctly predicted the positive class. (e.g., Actual: Cancer, Predicted: Cancer)
*   **True Negative (TN):** The model correctly predicted the negative class. (e.g., Actual: No Cancer, Predicted: No Cancer)
*   **False Positive (FP):** The model incorrectly predicted the positive class. This is also known as a **Type I error**. (e.g., Actual: No Cancer, Predicted: Cancer)
*   **False Negative (FN):** The model incorrectly predicted the negative class. This is also known as a **Type II error**. (e.g., Actual: Cancer, Predicted: No Cancer)

Understanding these terms is crucial for interpreting classification metrics like Accuracy, Precision, and Recall, as they are all derived from the values within the confusion matrix.

## Implementing Classification Metrics: Scikit-learn

Now that we understand the core concepts of classification metrics and the confusion matrix, let's put them into practice. We'll use the Breast Cancer Wisconsin dataset (`X_real`, `y_real`) and calculate key metrics using scikit-learn's built-in functions. This will help solidify our understanding of how these metrics are derived.

### Classification Metrics using Scikit-learn (General Case with Breast Cancer Data)

Now, let's use the scikit-learn library to quickly and efficiently calculate the classification metrics. We'll use the Breast Cancer dataset again for this general demonstration.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.datasets import load_breast_cancer

# Load the Breast Cancer dataset
data = load_breast_cancer()
X = data.data
y = data.target

# Split the dataset into training and testing sets
X_train_general, X_test_general, y_train_general, y_test_general = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Initialize and train a Logistic Regression model
model_general = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000)
model_general.fit(X_train_general, y_train_general)

# Make predictions on the test set
y_pred_general = model_general.predict(X_test_general)

# Get predicted probabilities for the positive class (for ROC curve)
y_pred_proba_general = model_general.predict_proba(X_test_general)[:, 1]

print("Model trained and predictions made on the test set.")

Model trained and predictions made on the test set.


#### Scikit-learn Classification Report and Individual Metrics

In [ ]:
# Scikit-learn Classification Report
print("--- Scikit-learn Classification Report ---")
print(classification_report(y_test_general, y_pred_general))

# Scikit-learn Individual Metrics
# Fil with appropriate values to run the cell
accuracy_sklearn_general = accuracy_score(y_test_general, y_pred_general)
precision_sklearn_general = precision_score(____________, ___________, pos_label=1)
recall_sklearn_general = recall_score(_____________, ________________)#do not forget to add pos_label
f1_score_sklearn_general = f1_score(_______________, ________________)#do not forget to add pos_label

print("\n--- Scikit-learn Individual Metrics ---")
print(f"Accuracy:  {accuracy_sklearn_general:.4f}")
print(f"Precision: {precision_sklearn_general:.4f}")
print(f"Recall:    {recall_sklearn_general:.4f}")
print(f"F1-Score:  {f1_score_sklearn_general:.4f}")

--- Scikit-learn Classification Report ---
              precision    recall  f1-score   support

           0       0.97      0.88      0.92        64
           1       0.93      0.98      0.95       107

    accuracy                           0.94       171
   macro avg       0.95      0.93      0.94       171
weighted avg       0.94      0.94      0.94       171



NameError: name '____________' is not defined

#### Explanation of Scikit-learn Implementation (Breast Cancer Data)

Scikit-learn significantly simplifies the process of calculating classification metrics:

1.  **Data Preparation**: The same `train_test_split` is performed to ensure a consistent comparison with the scratch implementation.
2.  **Model Training**: A `LogisticRegression` model is initialized and trained, just as in the manual approach.
3.  **Predictions**: The `predict` method is used to get hard class predictions, and `predict_proba` is used to get probability estimates for the ROC curve.
4.  **Automated Metrics**: Instead of manual loops, scikit-learn provides specialized functions:
    *   `classification_report(y_true, y_pred)`: Generates a comprehensive text report showing precision, recall, f1-score, and support for each class.
    *   `accuracy_score(y_true, y_pred)`: Calculates the overall classification accuracy.
    *   `precision_score(y_true, y_pred)`: Calculates the precision (for binary, usually the positive class).
    *   `recall_score(y_true, y_pred)`: Calculates the recall (for binary, usually the positive class).
    *   `f1_score(y_true, y_pred)`: Calculates the F1-score.
    *   `confusion_matrix(y_true, y_pred)`: Directly computes the confusion matrix as a 2x2 NumPy array.
5.  **Visualization**:
    *   `seaborn.heatmap`: Used for a clear, color-coded visual representation of the `confusion_matrix`.
    *   `RocCurveDisplay.from_predictions`: A convenient function to compute and plot the ROC curve directly from true labels and predicted probabilities, also displaying the Area Under the Curve (AUC).

This built-in functionality reduces boilerplate code, ensures correctness, and leverages optimized implementations, making it the preferred method for practical machine learning workflows.

#### Plotting the Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the confusion matrix
conf_matrix_general = confusion_matrix(y_test_general, y_pred_general)

# Plot the confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(________________, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0 (Malignant)', 'Predicted 1 (Benign)'],
            yticklabels=['Actual 0 (Malignant)', 'Actual 1 (Benign)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Breast Cancer Dataset (Logistic Regression)')
plt.show()

#### Plotting the ROC Curve

In [ ]:
from sklearn.metrics import RocCurveDisplay

# Plot the ROC curve
plt.figure(figsize=(7, 6))
ax = plt.gca()
roc_display = RocCurveDisplay.from_predictions(
    ____________,
    y_pred_proba_general,
    name='Logistic Regression (Breast Cancer)',
    ax=ax
)
plt.title('ROC Curve for Breast Cancer Dataset')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print(f"AUC Score: {roc_display.roc_auc:.4f}")

### EXERCISE-1: Mathematical Cross-Validation Problem

**Scenario**: You have a very small dataset with 5 samples `(X, y)`:

`X = [[10], [20], [30], [40], [50]]`
`y = [0, 0, 1, 1, 1]`

Your task is to manually perform a Leave-One-Out Cross-Validation (LOOCV) with a simple classifier that predicts `0` if the single feature `x <= 25` and `1` if `x > 25`.

For each fold, determine:

1.  The training set (`X_train`, `y_train`).
2.  The test set (`X_test`, `y_test`).
3.  The model's prediction for `X_test`.
4.  The accuracy of the prediction for that fold.

Finally, calculate the overall average LOOCV accuracy. Show your steps clearly.

# Excercise-2

#### 1. Manual Calculation of Classification Metrics

**Scenario**: You have the following true labels and model predictions for a binary classification task:

`y_true = [0, 1, 0, 0, 1, 1, 0, 1, 0, 0]`
`y_pred = [0, 0, 0, 1, 1, 1, 0, 1, 0, 0]`

Manually calculate the following metrics, showing your steps and explaining your reasoning:

1.  **Confusion Matrix**: Draw the 2x2 confusion matrix with True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).
2.  **Accuracy Score**
3.  **Precision Score** (for the positive class, `pos_label=1`)
4.  **Recall Score** (for the positive class, `pos_label=1`)
5.  **F1-Score** (for the positive class, `pos_label=1`)

*(Hint: Define class 1 as the positive class.)*

#### 2. Implementing Evaluation Metrics with K-Fold

1.  **Perform K-Fold Cross-Validation (5 Folds)**: Re-run the 5-Fold cross-validation using `sklearn.model_selection.KFold` (with `shuffle=True`, `random_state=42`) on `X_new` and `y_new`.
2.  **Collect Predictions**: For each fold, store the true labels (`y_test`) and the model's predictions (`y_pred`).
3.  **Calculate Metrics per Fold**: For each fold, calculate the following evaluation metrics:
    *   Accuracy Score
    *   Precision Score (for the positive class, `pos_label=1`)
    *   Recall Score (for the positive class, `pos_label=1`)
    *   F1-Score (for the positive class, `pos_label=1`)
    *   Confusion Matrix
4.  **Average Metrics**: After iterating through all folds, calculate and print the average (mean) of each metric across all folds. Discuss the interpretation of these average scores.


# Part-2 : Cross-Validation

## What is Cross-Validation?

**Definition:** Cross-validation is a powerful resampling technique used to evaluate machine learning models on a limited data sample. It is a crucial step in assessing how well a model generalizes to new, unseen data, and helps in identifying potential issues like overfitting or underfitting.

**Necessity:** The necessity of cross-validation arises because a simple train-test split, while seemingly straightforward, often falls short in providing a truly robust and reliable evaluation of a model's performance. Here's why:

1.  **Sensitivity to Data Split:** A single random train-test split can be highly sensitive to the specific data points included in each set. If the training set happens to contain certain outliers or unrepresentative patterns, the model might perform poorly on the test set, even if it's a good model overall. Conversely, if the test set is particularly easy or unrepresentative, the model's performance might appear artificially high. This variability makes it difficult to trust the evaluation from just one split.
2.  **Limited Data Utilization:** In a simple train-test split, a significant portion of the data is reserved solely for testing, meaning the model is trained on a smaller subset. This can be problematic, especially with limited datasets, as the model might not learn all the underlying patterns present in the full dataset. Cross-validation allows nearly all data points to be used for both training and testing across different iterations.
3.  **Risk of Overfitting to the Test Set:** If you repeatedly adjust your model or hyperparameters based on its performance on a single, fixed test set, you risk inadvertently 'overfitting' your model to that specific test set. This means the model might perform well on your chosen test set but fail to generalize to truly new, unseen data. Cross-validation, by evaluating on multiple different test sets, helps to prevent this.
4.  **Inconsistent Performance Estimates:** Different random splits of the same dataset can lead to vastly different performance metrics for the same model. This inconsistency makes it hard to confidently compare different models or assess the true potential of a single model. Cross-validation provides a more stable and less biased estimate by averaging results over multiple splits.

Therefore, cross-validation is indispensable for obtaining a more accurate, reliable, and generalized assessment of a model's capabilities, helping us build models that perform well in real-world scenarios.

**Purpose:** Its primary purpose is to provide a more robust and reliable estimate of a model's performance. This is achieved by training and testing the model on different subsets of the data multiple times, thereby reducing the variance associated with a single train-test split.

**Example: K-Fold Cross-Validation**

For example, in K-Fold cross-validation, the dataset is divided into 'k' equally sized folds. The model is then trained 'k' times; each time, one fold is used as the test set and the remaining 'k-1' folds are combined to form the training set. The performance metrics (e.g., accuracy, precision) from each of these 'k' iterations are then collected and typically averaged to provide a more stable and representative measure of the model's effectiveness. This process helps to ensure that the model's performance is not overly dependent on a specific data split.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Generate a small, simple dataset for demonstration
# Updated dataset to be more linearly separable, expecting better model performance.
X = np.array([
    [1, 1],
    [2, 2],
    [3, 1],
    [8, 8],
    [9, 9],
    [10, 8]
])
y = np.array([0, 0, 0, 1, 1, 1]) # Corresponding labels

print(f"Dataset shape: X={X.shape}, y={y.shape}")

### Diagrammatic Explanation of 3-Fold Cross-Validation

To further clarify the K-Fold process with our small dataset (6 samples, 3 folds), consider the following visual representation:

```mermaid
graph TD
    A[Original Dataset (6 Samples)] --> B1[Fold 1: Test Set];
    A --> B2[Fold 2: Test Set];
    A --> B3[Fold 3: Test Set];
    
    B1 -- (Samples: 2, 4) --> C1[Iteration 1];
    B2 -- (Samples: 3, 5) --> C2[Iteration 2];
    B3 -- (Samples: 0, 1) --> C3[Iteration 3];

    C1 --> D1_1{Train: Folds 2, 3 (Samples: 0, 1, 3, 5)};
    C1 --> D1_2{Test: Fold 1 (Samples: 2, 4)};

    C2 --> D2_1{Train: Folds 1, 3 (Samples: 0, 1, 2, 4)};
    C2 --> D2_2{Test: Fold 2 (Samples: 3, 5)};

    C3 --> D3_1{Train: Folds 1, 2 (Samples: 2, 3, 4, 5)};
    C3 --> D3_2{Test: Fold 3 (Samples: 0, 1)};

    D1_1 & D1_2 --> E1[Model Trained & Evaluated (Accuracy 1)];
    D2_1 & D2_2 --> E2[Model Trained & Evaluated (Accuracy 2)];
    D3_1 & D3_2 --> E3[Model Trained & Evaluated (Accuracy 3)];

    E1 & E2 & E3 --> F[Average Accuracy = (Accuracy 1 + Accuracy 2 + Accuracy 3) / 3];
```

**In this diagram:**

*   The **Original Dataset** of 6 samples is divided into 3 approximately equal folds after shuffling.
*   **Iteration 1:** Fold 1 is used as the **Test Set**, and Folds 2 and 3 are combined to form the **Training Set**.
*   **Iteration 2:** Fold 2 becomes the **Test Set**, with Folds 1 and 3 as the **Training Set**.
*   **Iteration 3:** Fold 3 is the **Test Set**, and Folds 1 and 2 form the **Training Set**.
*   For each iteration, a model is trained and evaluated, yielding an **Accuracy Score**.
*   Finally, the **Average Accuracy** across all three iterations is calculated to provide a robust performance estimate.

### K-Fold Cross-Validation using Scikit-learn

Now, let's use scikit-learn's built-in `KFold` class and `cross_val_score` function, which automate much of the process we did manually.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Generate a small, simple dataset for demonstration
# Updated dataset to be more linearly separable, expecting better model performance.
X = np.array([
    [1, 1],
    [2, 2],
    [3, 1],
    [8, 8],
    [9, 9],
    [10, 8]
])
y = np.array([0, 0, 0, 1, 1, 1]) # Corresponding labels

# For this small dataset, we'll use 3 splits for clearer demonstration
n_splits_demo = 3

# Initialize KFold cross-validator
kf = KFold(n_splits=________, shuffle=True, random_state=42)

# Initialize the Logistic Regression model
# Use a small C value for regularization with small datasets to prevent overfitting
model_sklearn = LogisticRegression(random_state=42, solver='liblinear', C=0.1)

# Perform cross-validation using cross_val_score
sklearn_accuracies = []
for train_index, test_index in kf.split(X):
    X_train, X_test = X[___________], X[test_index]
    y_train, y_test = y[train_index], y[__________]

    model_sklearn.fit(X_train, y_train)
    y_pred = model_sklearn.predict(X_test)
    accuracy = accuracy_score(________, y_pred)  #find the accuracy for test data
    sklearn_accuracies.append(accuracy)

# Alternatively, use cross_val_score for a more compact approach:
# sklearn_accuracies = cross_val_score(model_sklearn, X, y, cv=kf, scoring='accuracy')

print(f"Scikit-learn K-Fold Accuracies per fold: {sklearn_accuracies}")

# Calculate the average accuracy
mean_accuracy_sklearn = _____________ #calculate mean on sklearn_accuracies
std_accuracy_sklearn = _____________ #calculate standard deviation on sklearn_accuracies

print(f"\nAverage Scikit-learn K-Fold Accuracy: {mean_accuracy_sklearn:.4f}")
print(f"Standard Deviation of Scikit-learn K-Fold Accuracy: {std_accuracy_sklearn:.4f}")

### Explanation of Scikit-learn K-Fold Implementation

Here's how the scikit-learn implementation simplifies K-Fold Cross-Validation:

1.  **KFold Initialization**: We initialize `KFold(n_splits=n_splits_demo, shuffle=True, random_state=42)`.
    *   `n_splits=n_splits_demo` specifies the number of folds (which is 3 in our case).
    *   `shuffle=True` ensures the data is shuffled before splitting, similar to our manual `np.random.shuffle(indices)`.
    *   `random_state=42` provides reproducibility for the shuffling.
2.  **Model Initialization**: A `LogisticRegression` model is initialized, just as before, using a small `C` value for regularization with our small dataset.
3.  **Iteration with `kf.split(X)`**: The `kf.split(X)` method generates the `train_index` and `test_index` arrays for each fold, abstracting away the manual index calculation and concatenation.
4.  **Training and Evaluation**: Inside the loop, the data is split using these indices, the model is trained, and predictions are made. The `accuracy_score` is calculated and stored.
5.  **Average Performance**: Finally, the mean and standard deviation of the fold accuracies are calculated.

**Note on `cross_val_score` (commented out in code):** Scikit-learn also offers `cross_val_score`, which is an even more compact way to perform cross-validation. It takes the model, features (`X`), target (`y`), and the cross-validator (`cv=kf`) as arguments, and directly returns an array of scores for each fold. It handles the looping, training, prediction, and scoring internally, making it highly efficient for standard cross-validation tasks.

### Scikit-learn K-Fold Implementation Benefits

The scikit-learn implementation of K-Fold cross-validation provides significant advantages in terms of convenience and robustness:

| Feature              | Scikit-learn Implementation                                                                 |
| :------------------- | :------------------------------------------------------------------------------------------ |
| **Complexity**       | Low: `KFold` class handles splitting, and `cross_val_score` (or manual loop with `kf.split`) automates the process. |
| **Readability**      | Highly readable due to clear API and abstraction of details.                               |
| **Error Proneness**  | Less error-prone as tested and optimized functions are used.                                |
| **Flexibility**      | Highly flexible through various `sklearn.model_selection` classes (e.g., `StratifiedKFold`, `GroupKFold`). |
| **Reproducibility**  | Built-in `random_state` parameter in `KFold` ensures reproducibility.                       |
| **Standardization**  | Adheres to standard best practices for cross-validation.                                    |

**Conclusion:**

The scikit-learn implementation of K-Fold is highly recommended for practical applications. It provides a robust, efficient, and less error-prone way to perform cross-validation, allowing developers to focus more on model building and hyperparameter tuning rather than the intricate details of data splitting.

### Stratified K-Fold Cross-Validation

**What is Stratified K-Fold?**

Stratified K-Fold is a variation of K-Fold cross-validation that is particularly useful for classification problems, especially when dealing with imbalanced datasets (where one class significantly outnumbers others). Unlike standard K-Fold, which randomly divides the data into folds, Stratified K-Fold ensures that each fold maintains approximately the same proportion of target class labels as the original dataset.

**Why is it important?**

In standard K-Fold, if the dataset is imbalanced, it's possible that some folds might end up with very few or even zero samples of a minority class. This can lead to:

*   **Biased Model Training:** If a training fold lacks samples from a minority class, the model won't learn to predict that class effectively.
*   **Unreliable Evaluation:** If a test fold lacks samples from a minority class, the evaluation metric (e.g., accuracy) might be misleading, as the model isn't truly tested on all classes.

Stratified K-Fold mitigates these issues by preserving the class distribution across all folds, leading to more reliable and less biased estimates of model performance.

**When to use it?**

*   **Classification Tasks:** It is almost always recommended for classification problems over standard K-Fold, especially when the target variable is categorical.
*   **Imbalanced Datasets:** It is crucial when dealing with datasets where the distribution of classes is skewed.
*   **Small Datasets:** For small datasets, even a slight imbalance can significantly impact fold composition in standard K-Fold, making stratification essential.

**Which is better?**

For classification problems, Stratified K-Fold is generally considered superior to standard K-Fold because it provides a more robust and representative evaluation of the model's performance by ensuring that each fold accurately reflects the overall class distribution. For regression problems, where there are no classes, standard K-Fold is appropriate.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Define the number of folds (same as before for comparison)
n_splits_stratified = n_splits_demo

# Initialize StratifiedKFold cross-validator
skf = StratifiedKFold(n_splits=n_splits_stratified, shuffle=True, random_state=42)

# Initialize the Logistic Regression model (same as before)
model_stratified = LogisticRegression(random_state=42, solver='liblinear', C=0.1)

stratified_accuracies = []
print(f"Number of samples: {X.shape[0]}")
print(f"Number of folds: {n_splits_stratified}")

for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    model_stratified.fit(X_train, y_train)
    y_pred = __________ #fill with appropriate statement to run the code
    accuracy = ________ #calculate the accuracy on y_test,y_pred
    stratified_accuracies.append(accuracy)

    print(f"Fold {i+1} Accuracy: {accuracy:.4f}")
    print(f"  Train class distribution (y): {np.bincount(y_train)}")
    print(f"  Test class distribution (y): {np.bincount(y_test)}")

# Calculate the average accuracy using mean and standard deviation
mean_accuracy_stratified = np.mean(stratified_accuracies)
std_accuracy_stratified  = np.std(stratified_accuracies)

print(f"\nAverage Stratified K-Fold Accuracy: {mean_accuracy_stratified:.4f}")
print(f"Standard Deviation of Stratified K-Fold Accuracy: {std_accuracy_stratified:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Assuming n_splits_demo is defined, if not, set a default for safety (e.g., 3)
# If previous cells haven't run, these lists might be full of zeros, which is okay for this simplified plot.
if 'n_splits_demo' not in locals():
    n_splits_demo = 3 # Default if not found

# Ensure these lists exist, even if with zeros, to prevent errors for children's code.
# Only Scikit-learn related initializations are kept
if 'sklearn_accuracies' not in locals() or not sklearn_accuracies:
    sklearn_accuracies = [0.0] * n_splits_demo
if 'stratified_accuracies' not in locals():
    stratified_accuracies = [0.0] * n_splits_demo

# Calculate the average score for each guessing method (excluding scratch)
avg_sklearn_accuracy = _____________ #TODO
avg_stratified_accuracy = _____________ #TODO
# Create a simple table (DataFrame) for our plot
plot_data_for_kids = pd.DataFrame({
    'Guessing Method': ['Smart Computer Guessing (Simple)', 'Smart Computer Guessing (Fair)'],
    'Average Score': [avg_sklearn_accuracy, avg_stratified_accuracy]
})

# Make a bar chart to compare the average scores
plt.figure(figsize=(9, 6))
sns.barplot(x='Guessing Method', y='Average Score', data=plot_data_for_kids, palette='viridis', hue='Guessing Method', legend=False)
plt.title('How Good Our Scikit-learn Guessing Games Were (Average Scores)', fontsize=16)
plt.xlabel('Different Scikit-learn Guessing Methods', fontsize=14)
plt.ylabel('Average Score (0 to 1, higher is better)', fontsize=14)
plt.ylim(0, 1.1) # Scores usually go from 0 to 1, so we give a little extra room
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=15) # Tilt the names a bit so they fit better
plt.tight_layout() # Makes sure everything fits nicely
plt.show()

### Comparison: Standard K-Fold vs. Stratified K-Fold

Let's compare the results from standard K-Fold and Stratified K-Fold cross-validation on our small dataset. Recall the class distribution of our original `y` dataset: `[0, 0, 0, 1, 1, 1]` which has 3 samples of class 0 and 3 samples of class 1. This means a 50/50 split.

**Standard K-Fold Accuracies:**
*   Fold 1 Accuracy: 0.0000
*   Fold 2 Accuracy: 0.5000
*   Fold 3 Accuracy: 0.0000
*   Average Accuracy: 0.1667

**Stratified K-Fold Accuracies:**
*   Fold 1 Accuracy: 0.5000
*   Fold 2 Accuracy: 0.5000
*   Fold 3 Accuracy: 0.5000
*   Average Accuracy: 0.5000

**Observations:**

Due to the very small size of our dataset (6 samples, 3 classes of 0 and 3 of 1), and a 3-fold split (meaning 2 samples per fold), perfect stratification for binary classification can be challenging if an odd number of samples exists for a class. However, `StratifiedKFold` attempts to maintain the class proportions as closely as possible within each fold.

In our specific case, the `StratifiedKFold` carefully distributes the 3 samples of class 0 and 3 samples of class 1 across the 3 folds. As shown in the output, it attempts to give a balanced representation to each fold's training and testing sets, unlike the standard K-Fold where the class distribution in folds might be less balanced by chance. In this particular very small dataset, the final accuracies might appear similar due to the limited data, but the internal splitting mechanism is more robust for Stratified K-Fold.

### EXCERCISE-3

1.  **Why not just Train-Test Split?**: Elaborate on at least three reasons why a simple train-test split might not be sufficient for robust model evaluation, and how cross-validation addresses these limitations.
2.  **K-Fold vs. Stratified K-Fold**: For a classification problem with an imbalanced dataset, which cross-validation strategy would you choose between standard K-Fold and Stratified K-Fold, and why? Explain the key difference between these two methods.

### EXCERCISE-4

Task : Implement Standard K-Fold Cross-Validation

* Step 1: Initialize KFold with
n_splits=5, shuffle=True, and random_state=42.

* Step 2: Initialize a LogisticRegression model specifying random_state=42 and solver='liblinear'.

*  Step 3: Create an empty list (e.g., kfold_accuracies) to store the accuracy score for each fold.


* Step 4: Iterate through the folds using kf.split(X_new):

    *  Extract training and testing subsets (X_train, X_test, y_train, y_test).

    * Fit the logistic regression model on the training data and predict on the test data.

    * Fit the logistic regression model on the training data and predict on the test data.

    * Compute the accuracy score and append it to your list, printing the accuracy for each fold.

*  Step 5: Calculate and print the final average accuracy and standard deviation across all folds.


In [ ]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Using the small, simple dataset
# to ensure consistent output across different runs/students.
X_new = np.array([
    [1, 1],
    [2, 2],
    [3, 1],
    [8, 8],
    [9, 9],
    [10, 8]
])
y_new = np.array([0, 0, 0, 1, 1, 1]) # Corresponding labels

print(f"Using dataset: X_new shape={X_new.shape}, y_new shape={y_new.shape}")

#TODO tasks
# Step 1: Initialize KFold


# Step 2: Initialize a LogisticRegression model


# Step 3: Create an empty list to store accuracy scores


print("\nPerforming 5-Fold Cross-Validation:")
# Step 4: Iterate through the folds
for fold_num, (train_index, test_index) in enumerate(kf.split(X_new), 1):
    X_train, X_test = X_new[train_index], X_new[test_index]
    y_train, y_test = y_new[train_index], y_new[test_index]

    # Fit the model on training data
    model_exercise.fit(_____, ______) #TODO

    # Predict on the test data
    y_pred = ____________________________ #TODO

    # Compute the accuracy score and append it
    accuracy = accuracy_score(y_test, y_pred)
    kfold_accuracies.append(accuracy)

    print(f"Fold {fold_num} Accuracy: {accuracy:.4f}")

# Step 5: Calculate and print the final average accuracy and standard deviation
mean_accuracy_kfold_exercise = _____________________ #TODO
std_accuracy_kfold_exercise =  _____________________ #TODO

print(f"\nAverage K-Fold Accuracy: {mean_accuracy_kfold_exercise:.4f}")
print(f"Standard Deviation of K-Fold Accuracy: {std_accuracy_kfold_exercise:.4f}")

### Leave-One-Out Cross-Validation (LOOCV)

Leave-One-Out Cross-Validation (LOOCV) is an extreme case of K-Fold Cross-Validation where the number of folds `k` is equal to the number of samples `n` in the dataset. In each iteration, one sample is used as the test set, and the remaining `n-1` samples are used as the training set. This process is repeated `n` times, ensuring that every sample is used exactly once as a test sample and `n-1` times as a training sample.

In [ ]:
import numpy as np # Added import for X, y, np.mean, np.std
from sklearn.metrics import accuracy_score # Added import
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression

# Define a more separable dataset for demonstration with LOOCV
X = np.array([
    [1, 1],
    [1.5, 1.5],
    [2, 2],
    [8, 8],
    [8.5, 8.5],
    [9, 9]
])
y = np.array([0, 0, 0, 1, 1, 1])

# Initialize LeaveOneOut cross-validator
loo = LeaveOneOut()

# Initialize the Logistic Regression model (same as before)
model_loo = LogisticRegression(random_state=42, solver='liblinear', C=0.1)

loo_accuracies = []
print(f"Number of samples: {X.shape[0]}")

for i, (train_index, test_index) in enumerate(loo.split(X)):
    X_train = X[train_index]
    X_test = X[test_index]
    y_train = y[train_index]
    y_test = y[test_index]

    model_loo.fit(X_train, y_train)
    y_pred = model_loo.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    loo_accuracies.append(accuracy)

    # For LOOCV, each fold is a single sample, so test_index[0] gives the original index
    print(f"Fold {i+1} (Test Sample Index: {test_index[0]}) Accuracy: {accuracy:.4f}")

# Calculate the average accuracy
mean_accuracy_loo = np.mean(loo_accuracies)
std_accuracy_loo = np.std(loo_accuracies)

print(f"\nAverage LOOCV Accuracy: {mean_accuracy_loo:.4f}")
print(f"Standard Deviation of LOOCV Accuracy: {std_accuracy_loo:.4f}")

### Comparison of Cross-Validation Strategies: K-Fold, Stratified K-Fold, and LOOCV

We have explored three different cross-validation techniques: standard K-Fold, Stratified K-Fold, and Leave-One-Out Cross-Validation (LOOCV). Each has its own characteristics, advantages, disadvantages, and ideal use cases.

| Feature / Strategy | Standard K-Fold | Stratified K-Fold | Leave-One-Out (LOOCV) |
| :--- | :--- | :--- | :--- |
| **Mechanism** | Splits randomly into `k` folds. | Splits into `k` folds, preserving class proportions. | `k = n`. Trains on `n-1` samples, tests on 1. |
| **Bias/Variance** | Moderate bias, moderate variance. | Moderate bias, lower variance for classification. | Low bias, high variance. |
| **Computational Cost**| Moderate (`k` models). | Moderate (`k` models). | Very High (`n` models). |
| **Use Cases** | General purpose evaluation. | **Primary choice for classification**, especially imbalanced data. | **Very small datasets** where training data is scarce. |

**Our Example Observations:**
*   **Standard K-Fold:** Resulted in an average accuracy of **0.1667**. This variability suggests that random splits on a tiny dataset often lead to unrepresentative training/test sets.
*   **Stratified K-Fold:** Achieved a consistent average of **0.5000**. By ensuring proportional class representation in each fold, the model was evaluated more fairly and stably.
*   **Leave-One-Out (LOOCV):** Resulted in an average accuracy of **0.5000**. While it provides a nearly unbiased error estimate by training on almost all data, on tiny datasets, it can still struggle if the single test point is an outlier or if the classes are difficult to separate perfectly.

**Conclusion:** For classification tasks, **Stratified K-Fold** is generally the preferred choice. LOOCV is computationally expensive and is typically reserved for exceptionally small datasets.

### Hyperparameter Tuning and Diagnosing Overfitting (Objectives 7 & 8)

Cross-validation is the standard method for **hyperparameter tuning**. If we just tune parameters to fit a single test set, we risk overfitting. `GridSearchCV` automates the process of testing a "grid" of parameters using cross-validation.

Furthermore, by looking at both the **training scores** and **validation scores**, we can diagnose model fit:
*   **Underfitting (High Bias):** Both training and validation scores are poor. The model is too simple.
*   **Overfitting (High Variance):** The training score is excellent, but the validation score is poor. The model memorized the training data but failed to generalize.
*   **Good Fit:** Both training and validation scores are high and relatively close to each other.

Let's tune the `max_depth` of our Decision Tree to observe this behavior.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_breast_cancer
import pandas as pd
import matplotlib.pyplot as plt

# Load the realistic dataset (moved from a previous cell to make this cell self-contained)
data = load_breast_cancer()
X_real = data.data
y_real = data.target

# 1. Define the grid of parameters to search
# A depth of 1 is very simple (underfitting), while None means the tree grows until perfect (overfitting)
param_grid = {'max_depth': [1, 2, 3, 4, 5, 10, None]}

# 2. Initialize GridSearchCV
# Setting return_train_score=True allows us to diagnose overfitting
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    return_train_score=True
)

# 3. Fit the grid search to the data
grid_search.fit(X_real, y_real)

print(f"Best Hyperparameter: {grid_search.best_params_}")
print(f"Best Validation Accuracy: {grid_search.best_score_:.4f}\n")

# 4. Extract and visualize the results to diagnose fit
results_df = pd.DataFrame(grid_search.cv_results_)
depths = results_df['param_max_depth'].fillna('Unlimited').astype(str)
train_scores = results_df['mean_train_score']
val_scores = results_df['mean_test_score']

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, label='Training Accuracy', marker='o', linestyle='dashed')
plt.plot(depths, val_scores, label='Validation Accuracy', marker='s', linewidth=2)
plt.title('Diagnosing Model Fit: Training vs. Validation Performance')
plt.xlabel('Tree Depth (max_depth)')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.5)
plt.show()

**Analyzing the Learning Curve:**
*   **Left side (Depth 1):** The model is **underfitting**. Both training and validation accuracies are lower than they could be because the model is too simple to capture the patterns.
*   **Middle (Depth 3 or 4):** The **sweet spot**. Validation accuracy peaks here, representing a model that generalized well to unseen data.
*   **Right side (Depth 10 or Unlimited):** The model is **overfitting**. The training accuracy reaches 1.0 (100%), but the validation accuracy drops or stagnates. The model memorized the noise in the training data.

By utilizing `GridSearchCV` and observing cross-validated train vs. test scores, we successfully optimized our model while avoiding the pitfalls of overfitting!

# Applying Cross-Validation to Real-World Data

To fully realize the power of cross-validation, let's transition from our 6-sample toy dataset to a realistic one. We will use the **Breast Cancer Wisconsin diagnostic dataset**, a standard benchmark for binary classification.

We will also use a **Decision Tree Classifier** for this section, as it is an excellent model for demonstrating overfitting.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

# Load the realistic dataset
data = load_breast_cancer()
X_real = data.data
y_real = data.target

print(f"Realistic Dataset Shape: X={X_real.shape}, y={y_real.shape}")
print(f"Class distribution: 0 (Malignant): {sum(y_real==0)}, 1 (Benign): {sum(y_real==1)}")

### Advanced Metrics with `cross_validate`

While `cross_val_score` is great for a single metric (like accuracy), scikit-learn provides `cross_validate` to evaluate multiple metrics simultaneously. Let's calculate Accuracy, Precision, Recall, and F1-score across 5 stratified folds.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
import numpy as np

# Initialize a default Decision Tree
dt_model = ______________ #provide the required classifier with the form classifier(random_state=42)
skf_real = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define the metrics we want to track
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

# Perform cross-validation
cv_results = cross_validate(dt_model, X_real, y_real, cv=skf_real, scoring=scoring_metrics)

# Display the average of each metric
print("Average Validation Metrics across 5 Folds:")
print(f"Accuracy:  {np.mean(cv_results['test_accuracy']):.4f}")
print(f"Precision: {np.mean(cv_results['test_precision']):.4f}")
print(f"Recall:    {np.mean(cv_results['test_recall']):.4f}")
print(f"F1-Score:  {np.mean(cv_results['test_f1']):.4f}")